# 03 — Training Model Unsupervised Learning (Clustering Kondisi Wafer)

**Proyek:** SmartWafer — Klasifikasi Defect & Clustering Kondisi Wafer
**Modul:** Unsupervised Learning — ML Engineer 2
**Dataset:** `semiconductor_wafer_defect_featured.csv` (hasil `preprocessing.ipynb`)

**Tujuan.** Mengelompokkan wafer berdasarkan parameter proses menjadi beberapa **rezim operasi**, membandingkan tiga algoritma clustering (**K-Means, Agglomerative, DBSCAN**), memilih model terbaik dengan aturan yang eksplisit, lalu menandai wafer yang jauh dari pusat rezimnya sebagai **anomali**.

**Output (3 file, diunduh sebagai `clustering_output.zip`):**

| File | Isi |
|---|---|
| `best_model.joblib` | Model terbaik siap pakai: pipeline `StandardScaler → KMeans` + `PCA` (visualisasi) + daftar fitur |
| `model_comparison.csv` | Tabel perbandingan semua algoritma (metrik internal, noise, stabilitas, defect tertangkap, alasan terpilih/tidak) |
| `model_metadata.json` | Profil & nama cluster, ambang anomali, metrik, evaluasi eksternal, sampel titik PCA untuk grafik web |

> Jalankan dari atas ke bawah: **Runtime → Run all**. Saat diminta, upload file CSV.

## 1. Import library & konfigurasi

In [ ]:
import os, json, shutil, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
from scipy.cluster.hierarchy import dendrogram, linkage

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ---------------- KONFIGURASI ----------------
RANDOM_STATE = 42
K_RANGE = range(1, 11)          # Elbow mulai K=1 (sesuai modul)
N_CLUSTERS = 3                  # K final (lihat alasan di bagian 6)
ANOMALY_PERCENTILE = 95         # jarak > persentil ini (per cluster) = anomali
STABILITY_SEEDS = [0, 1, 7, 21, 99, 123, 256, 512, 1024, 2024]

# Fitur input model. URUTAN INI WAJIB SAMA saat inference di aplikasi.
FEATURES = ["temperature_c", "pressure_torr", "gas_flow_sccm",
            "etch_rate_nm_min", "voltage_v", "current_ma"]

FEATURE_LABELS = {"temperature_c": "Suhu", "pressure_torr": "Tekanan",
                  "gas_flow_sccm": "Aliran Gas", "etch_rate_nm_min": "Etch Rate",
                  "voltage_v": "Tegangan", "current_ma": "Arus"}

OUTPUT_DIR = "clustering_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("scikit-learn versi:", sklearn.__version__, "-> samakan di requirements.txt aplikasi!")

## 2. Load dataset

Notebook mencari CSV di folder kerja; jika tidak ada dan berjalan di Colab, muncul tombol **upload**. Pilih `semiconductor_wafer_defect_featured.csv` (atau `..._clean.csv`).

In [ ]:
CANDIDATES = ["semiconductor_wafer_defect_featured.csv", "semiconductor_wafer_defect_clean.csv",
              "../data/processed/semiconductor_wafer_defect_featured.csv",
              "data/processed/semiconductor_wafer_defect_featured.csv"]
csv_path = next((p for p in CANDIDATES if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError("CSV tidak ditemukan. Letakkan file di folder yang sama dengan notebook.")

df = pd.read_csv(csv_path)
print("File:", csv_path, "| Ukuran:", df.shape)

missing_cols = [c for c in FEATURES if c not in df.columns]
assert not missing_cols, f"Kolom tidak ditemukan: {missing_cols}"
assert df[FEATURES].isnull().sum().sum() == 0, "Masih ada missing value pada fitur!"
df.head()

## 3. Pemilihan fitur

| Kolom | Dipakai? | Alasan |
|---|---|---|
| 6 parameter proses numerik | **Ya** | Menggambarkan kondisi operasi wafer |
| `defect_label` | **Tidak** | Label target; memakainya = *data leakage*. Hanya untuk **validasi eksternal** di akhir (diizinkan modul) |
| `wafer_id` | **Tidak** | Identifier unik |
| `electrical_power_mw` | **Tidak** | Turunan `voltage_v × current_ma` → informasi listrik terhitung dua kali (redundan) |
| `process_step` | **Tidak** (diuji di bagian 10) | Kategori **nominal**. LabelEncoder memberi urutan angka palsu (berdasarkan abjad); modul teori 12.2: LabelEncoder untuk ordinal, One-Hot untuk nominal |

**Outlier tidak di-capping** (berbeda dari saran umum modul 12.2-B): pada kasus ini outlier justru sinyal kondisi tidak lazim yang ingin dideteksi.

In [ ]:
X_raw = df[FEATURES].copy()
X_raw.describe().T.round(3)

## 4. Scaling (StandardScaler)

Algoritma clustering berbasis jarak Euclidean, sehingga skala fitur harus disamakan (tekanan ~760 vs tegangan ~5).

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
pd.DataFrame(X, columns=FEATURES).describe().T[["mean", "std"]].round(3)

## 5. Cek struktur data

Jika korelasi antarfitur ≈ 0 dan varians PCA terbagi rata, data cenderung **tidak memiliki cluster alami**, sehingga hasil clustering dibaca sebagai *segmentasi*.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(X_raw.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Korelasi Antarfitur")

evr = PCA(random_state=RANDOM_STATE).fit(X).explained_variance_ratio_
axes[1].bar(range(1, len(evr) + 1), evr * 100)
axes[1].plot(range(1, len(evr) + 1), np.cumsum(evr) * 100, marker="o", color="#DD8452", label="Kumulatif")
axes[1].axhline(80, ls="--", color="grey", lw=1, label="80%")
axes[1].set_xlabel("Komponen PCA"); axes[1].set_ylabel("Varians dijelaskan (%)")
axes[1].set_title("Explained Variance PCA"); axes[1].legend()
plt.tight_layout(); plt.show()

print("Explained variance per komponen (%):", np.round(evr * 100, 1))
print("Komponen untuk >80% varians:", int(np.argmax(np.cumsum(evr) >= 0.8) + 1))

## 6. Menentukan jumlah cluster (K) — K-Means

- **Elbow (WCSS/inertia):** cari "siku".
- **Silhouette:** -1..1, makin tinggi makin baik (< 0.25 = tidak ada struktur substansial).
- **Davies-Bouldin:** makin rendah makin baik. **Calinski-Harabasz:** makin tinggi makin baik.

In [ ]:
rows = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    row = {"k": k, "inertia": km.inertia_}
    if k >= 2:
        row.update(silhouette=silhouette_score(X, km.labels_),
                   davies_bouldin=davies_bouldin_score(X, km.labels_),
                   calinski_harabasz=calinski_harabasz_score(X, km.labels_))
    rows.append(row)
k_eval = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(k_eval.k, k_eval.inertia, "o-"); axes[0].set_title("Elbow Method (WCSS)")
s = k_eval.dropna()
axes[1].plot(s.k, s.silhouette, "o-", color="#55A868")
axes[1].axhline(0.25, ls="--", color="grey", lw=1, label="Batas 0.25"); axes[1].legend()
axes[1].set_title("Silhouette Score")
axes[2].plot(s.k, s.davies_bouldin, "o-", color="#C44E52"); axes[2].set_title("Davies-Bouldin Index")
for ax in axes: ax.set_xlabel("K"); ax.set_xticks(list(K_RANGE))
plt.tight_layout(); plt.show()

print("K dengan silhouette tertinggi:", int(s.loc[s.silhouette.idxmax(), "k"]))
k_eval.round(4)

### Keputusan K

Elbow landai tanpa siku tegas dan silhouette **datar ~0.10–0.11** untuk semua K, sehingga metrik internal **tidak menunjukkan K optimal yang tegas** (data tidak memiliki kelompok alami). Sesuai modul 9.7 ("jika tidak setuju → pertimbangkan konteks"), K dipilih berdasarkan **interpretabilitas**: **K = 3** rezim operasi yang masih mudah dijelaskan ke operator. Ubah `N_CLUSTERS` di sel konfigurasi jika kelompok memutuskan lain.

## 7. Algoritma 1 — K-Means

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, init="k-means++", n_init=10, random_state=RANDOM_STATE)
labels_km = kmeans.fit_predict(X)
print(f"K-Means (K={N_CLUSTERS}) — anggota per cluster:", np.bincount(labels_km))
print("Silhouette: %.4f | DBI: %.4f | CH: %.2f" % (silhouette_score(X, labels_km),
      davies_bouldin_score(X, labels_km), calinski_harabasz_score(X, labels_km)))

## 8. Algoritma 2 — Agglomerative Clustering (Ward)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx_dendro = rng.choice(len(X), size=1000, replace=False)
plt.figure(figsize=(14, 6))
dendrogram(linkage(X[idx_dendro], method="ward"), truncate_mode="lastp", p=25,
           leaf_rotation=90, leaf_font_size=9, show_leaf_counts=True)
plt.title("Dendrogram — Agglomerative (Ward), sampel 1000 wafer")
plt.ylabel("Jarak Euclidean"); plt.tight_layout(); plt.show()

agg = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward")
labels_agg = agg.fit_predict(X)
print(f"Agglomerative (K={N_CLUSTERS}) — anggota per cluster:", np.bincount(labels_agg))
print("Silhouette: %.4f | DBI: %.4f | CH: %.2f" % (silhouette_score(X, labels_agg),
      davies_bouldin_score(X, labels_agg), calinski_harabasz_score(X, labels_agg)))
print("Catatan: AgglomerativeClustering tidak punya .predict() untuk data baru.")

## 9. Algoritma 3 — DBSCAN

`min_samples` = jumlah fitur + 1 (aturan praktis modul 8.6). `eps` dipilih otomatis dari **titik siku k-distance graph** (titik dengan jarak terbesar dari garis lurus ujung-ke-ujung kurva yang dinormalisasi).

In [ ]:
MIN_SAMPLES = len(FEATURES) + 1
dist, _ = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X).kneighbors(X)
k_dist = np.sort(dist[:, -1])
xn = np.linspace(0, 1, len(k_dist))
yn = (k_dist - k_dist.min()) / (k_dist.max() - k_dist.min())
knee_idx = int(np.argmax(xn - yn))
EPS = float(k_dist[knee_idx])

plt.figure(figsize=(10, 5))
plt.plot(k_dist, lw=2)
plt.axhline(EPS, color="red", ls="--", label=f"eps (knee) = {EPS:.3f}")
plt.scatter([knee_idx], [EPS], color="red", zorder=3)
plt.xlabel("Titik data (diurutkan)"); plt.ylabel(f"Jarak ke tetangga ke-{MIN_SAMPLES}")
plt.title("k-Distance Graph — Menentukan eps DBSCAN"); plt.legend(); plt.tight_layout(); plt.show()

# Grid eksperimen (sesuai modul) untuk konteks
grid = []
for eps in [0.8, 1.0, 1.2, round(EPS, 3), 1.5, 1.8]:
    for ms in [3, 5, MIN_SAMPLES]:
        l = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
        grid.append({"eps": eps, "min_samples": ms,
                     "n_clusters": len(set(l)) - (1 if -1 in l else 0),
                     "noise_pct": round((l == -1).mean() * 100, 2)})
display(pd.DataFrame(grid))

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
labels_db = dbscan.fit_predict(X)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
print(f"DBSCAN terpilih: eps={EPS:.3f}, min_samples={MIN_SAMPLES} -> "
      f"{n_clusters_db} cluster, noise {(labels_db == -1).sum()} titik ({(labels_db == -1).mean()*100:.1f}%)")

## 10. Uji fitur: 6 fitur vs 7 fitur (+`process_step` LabelEncoder)

Studi kasus praktikum menambahkan `process_step` hasil LabelEncoder. Diuji di sini untuk membuktikan dampaknya.

In [ ]:
le = LabelEncoder()
X7_raw = X_raw.copy()
X7_raw["process_step_encoded"] = le.fit_transform(df["process_step"])
X7 = StandardScaler().fit_transform(X7_raw)
labels_km7 = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE).fit_predict(X7)

print("Mapping LabelEncoder (berdasarkan abjad):", dict(zip(le.classes_, range(len(le.classes_)))))
print("Silhouette 6 fitur: %.4f | 7 fitur: %.4f" % (silhouette_score(X, labels_km), silhouette_score(X7, labels_km7)))
print("\nSebaran process_step per cluster (7 fitur):")
display(pd.crosstab(labels_km7, df["process_step"]))
print("Cluster terbelah mengikuti KODE ANGKA abjad tahap proses -> artefak encoding, bukan pola fisik.")

## 11. Uji stabilitas K-Means (modul 16.10)

K-Means dijalankan ulang dengan random seed berbeda, lalu dibandingkan dengan model utama memakai **Adjusted Rand Index** (1 = identik, 0 = seperti acak).

In [ ]:
ari_seeds = [adjusted_rand_score(labels_km, KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=s).fit_predict(X))
             for s in STABILITY_SEEDS]
ari_7 = [adjusted_rand_score(labels_km7, KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=s).fit_predict(X7))
         for s in STABILITY_SEEDS]
print("ARI per seed (6 fitur):", np.round(ari_seeds, 3))
print("Rata-rata ARI 6 fitur: %.3f | 7 fitur: %.3f" % (np.mean(ari_seeds), np.mean(ari_7)))
print("random_state=42 menjamin REPRODUCIBLE, tetapi batas cluster TIDAK stabil -> konsekuensi data tanpa struktur alami.")

## 12. Deteksi anomali & evaluasi eksternal per algoritma

- **K-Means:** anomali = jarak ke centroid > persentil ke-`ANOMALY_PERCENTILE` per cluster.
- **DBSCAN:** anomali = titik **noise** (label -1).
- **Agglomerative:** tidak punya centroid/predict; dihitung dengan jarak ke rata-rata cluster (hanya untuk perbandingan).

`defect_label` **hanya** dipakai di sini untuk evaluasi. ⚠️ Hanya **7 defect dari 5000** wafer, jadi angka ini indikasi, bukan bukti statistik.

In [ ]:
y_defect = df["defect_label"].astype(int).values if "defect_label" in df.columns else None

def centroid_anomaly(X_, labels_, centers_, pct=ANOMALY_PERCENTILE):
    d = np.linalg.norm(X_ - centers_[labels_], axis=1)
    thr = {c: float(np.percentile(d[labels_ == c], pct)) for c in np.unique(labels_)}
    flag = np.array([d[i] > thr[labels_[i]] for i in range(len(d))])
    return d, thr, flag

def ext_eval(flag):
    if y_defect is None:
        return {}
    flagged = int(flag.sum()); caught = int(y_defect[flag].sum()); total = int(y_defect.sum())
    base = y_defect.mean(); rate = caught / flagged if flagged else 0.0
    return {"flagged_total": flagged, "defects_flagged": caught, "defects_total": total,
            "base_defect_rate": float(base), "flagged_defect_rate": float(rate),
            "lift": float(rate / base) if base else None}

dist_km, thr_km, flag_km = centroid_anomaly(X, labels_km, kmeans.cluster_centers_)
centers_agg = np.vstack([X[labels_agg == c].mean(axis=0) for c in range(N_CLUSTERS)])
_, _, flag_agg = centroid_anomaly(X, labels_agg, centers_agg)
flag_db = labels_db == -1

ext = {"K-Means": ext_eval(flag_km), "Agglomerative": ext_eval(flag_agg), "DBSCAN": ext_eval(flag_db)}
pd.DataFrame(ext).T.round(4)

## 13. Tabel perbandingan model → `model_comparison.csv`

In [ ]:
def internal_metrics(X_, labels_):
    mask = labels_ != -1                     # noise DBSCAN tidak ikut dihitung
    n = len(set(labels_[mask]))
    if n < 2:
        return dict(silhouette=np.nan, davies_bouldin=np.nan, calinski_harabasz=np.nan)
    return dict(silhouette=silhouette_score(X_[mask], labels_[mask]),
                davies_bouldin=davies_bouldin_score(X_[mask], labels_[mask]),
                calinski_harabasz=calinski_harabasz_score(X_[mask], labels_[mask]))

def row(name, config, n_feat, labels_, can_predict, stability, ext_, candidate, note):
    n_cl = len(set(labels_)) - (1 if -1 in labels_ else 0)
    r = {"algoritma": name, "konfigurasi": config, "jumlah_fitur": n_feat,
         "n_clusters": n_cl, "noise_pct": round((labels_ == -1).mean() * 100, 2)}
    r.update({k: (round(v, 4) if pd.notna(v) else np.nan) for k, v in internal_metrics(X if n_feat == 6 else X7, labels_).items()})
    r.update({"stabilitas_ari_mean": round(stability, 4) if stability is not None else np.nan,
              "dapat_prediksi_data_baru": can_predict,
              "anomali_ditandai": ext_.get("flagged_total"),
              "defect_tertangkap": ext_.get("defects_flagged"),
              "defect_total": ext_.get("defects_total"),
              "lift": round(ext_["lift"], 2) if ext_.get("lift") else np.nan,
              "kandidat": candidate, "catatan": note})
    return r

comparison = pd.DataFrame([
    row("K-Means", f"K={N_CLUSTERS}, n_init=10, random_state={RANDOM_STATE}", 6, labels_km, True,
        float(np.mean(ari_seeds)), ext["K-Means"], True, "Model utama; anomali = jarak ke centroid > P%d" % ANOMALY_PERCENTILE),
    row("Agglomerative", f"K={N_CLUSTERS}, linkage=ward", 6, labels_agg, False, None,
        ext["Agglomerative"], True, "Deterministik; tidak punya predict() untuk data baru"),
    row("DBSCAN", f"eps={EPS:.3f} (knee), min_samples={MIN_SAMPLES}", 6, labels_db, False, None,
        ext["DBSCAN"], True, "Anomali = noise; tidak punya predict() untuk data baru"),
    row("K-Means", f"K={N_CLUSTERS}, +process_step LabelEncoder", 7, labels_km7, True,
        float(np.mean(ari_7)), ext_eval(centroid_anomaly(X7, labels_km7,
            KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE).fit(X7).cluster_centers_)[2]),
        False, "Eksperimen fitur; cluster mengikuti kode abjad process_step (artefak encoding)"),
])

# ---- Aturan pemilihan model terbaik (eksplisit) ----
# 1) kandidat, 2) bisa memprediksi data baru (wajib untuk aplikasi web),
# 3) menghasilkan >= 2 cluster, 4) silhouette tertinggi di antara yang lolos.
eligible = comparison[(comparison.kandidat) & (comparison.dapat_prediksi_data_baru) & (comparison.n_clusters >= 2)]
best_idx = eligible["silhouette"].idxmax()
comparison["terpilih"] = comparison.index == best_idx

def reason(r):
    if r.terpilih: return "TERPILIH: lolos semua syarat & silhouette tertinggi"
    if not r.kandidat: return "Bukan kandidat (eksperimen fitur)"
    if not r.dapat_prediksi_data_baru: return "Gugur: tidak bisa memprediksi data baru"
    if r.n_clusters < 2: return "Gugur: < 2 cluster"
    return "Gugur: silhouette lebih rendah"
comparison["alasan"] = comparison.apply(reason, axis=1)

comparison.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)
best_row = comparison.loc[best_idx]
print("Model terbaik:", best_row.algoritma, "|", best_row.konfigurasi)
comparison

In [ ]:
# Visualisasi perbandingan metrik (3 algoritma utama, 6 fitur)
main = comparison[comparison.kandidat].set_index("algoritma")
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (col, title) in zip(axes, [("silhouette", "Silhouette (↑)"), ("davies_bouldin", "Davies-Bouldin (↓)"),
                                   ("calinski_harabasz", "Calinski-Harabasz (↑)"), ("defect_tertangkap", "Defect tertangkap (↑)")]):
    vals = main[col].fillna(0)
    bars = ax.bar(vals.index, vals.values, color=["#4C72B0", "#DD8452", "#55A868"])
    ax.set_title(title)
    for b, v, raw in zip(bars, vals.values, main[col].values):
        ax.annotate("N/A" if pd.isna(raw) else f"{v:.3f}" if v < 100 else f"{v:.0f}",
                    (b.get_x() + b.get_width() / 2, b.get_height()), ha="center", va="bottom", fontsize=9)
plt.suptitle("Perbandingan Algoritma Clustering", y=1.03); plt.tight_layout(); plt.show()

## 14. Visualisasi PCA 2D — tiga algoritma

PCA **hanya untuk visualisasi**; model tetap dilatih pada 6 fitur ter-scaling.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
print("Explained variance PC1+PC2: %.1f%% (modul: >60%% 'cukup baik' -> di sini rendah, tafsirkan hati-hati)"
      % (pca.explained_variance_ratio_.sum() * 100))

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))
for ax, (name, lab) in zip(axes, [("K-Means", labels_km), ("Agglomerative", labels_agg), ("DBSCAN", labels_db)]):
    noise = lab == -1
    ax.scatter(X_pca[~noise, 0], X_pca[~noise, 1], c=lab[~noise], cmap="viridis", s=6, alpha=0.5)
    if noise.any():
        ax.scatter(X_pca[noise, 0], X_pca[noise, 1], c="lightgrey", s=10, marker="x", label="Noise")
    if y_defect is not None:
        d = y_defect == 1
        ax.scatter(X_pca[d, 0], X_pca[d, 1], s=90, facecolors="none", edgecolors="red", lw=2, label="Defect (label asli)")
    ax.set_title(name); ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"); ax.legend(fontsize=8)
plt.suptitle("Perbandingan Hasil Clustering (PCA 2D)", y=1.02); plt.tight_layout(); plt.show()

## 15. Profiling & penamaan cluster (model terbaik)

Nama otomatis dari centroid z-score: dua fitur dengan penyimpangan terbesar (|z| ≥ 0.3).

In [ ]:
best_model = kmeans                                   # sesuai baris 'terpilih' di tabel
assert best_row.algoritma == "K-Means" and best_row.jumlah_fitur == 6, "Sesuaikan export jika model terbaik berubah"
labels = labels_km

centroids_z = pd.DataFrame(best_model.cluster_centers_, columns=FEATURES)
centroids_orig = pd.DataFrame(scaler.inverse_transform(best_model.cluster_centers_), columns=FEATURES)

def name_cluster(z, threshold=0.3, top_n=2):
    dev = z.abs().sort_values(ascending=False)
    parts = [f"{FEATURE_LABELS[f]} {'Tinggi' if z[f] > 0 else 'Rendah'}" for f in dev.index[:top_n] if dev[f] >= threshold]
    return " · ".join(parts) if parts else "Kondisi Rata-rata"

cluster_names = {}
for c in range(N_CLUSTERS):
    n = name_cluster(centroids_z.loc[c])
    cluster_names[c] = f"{n} (C{c})" if n in cluster_names.values() else n
for c, n in cluster_names.items():
    print(f"Cluster {c}: {n}  ({(labels == c).sum()} wafer)")

plt.figure(figsize=(9, 3 + 0.4 * N_CLUSTERS))
sns.heatmap(centroids_z.rename(columns=FEATURE_LABELS, index={c: f"C{c}: {n}" for c, n in cluster_names.items()}),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Profil Centroid Cluster (z-score)"); plt.tight_layout(); plt.show()

if "process_step" in df.columns:
    display(pd.crosstab(labels, df["process_step"], normalize="index").round(3))
centroids_orig.round(2).rename(index=cluster_names)

## 16. Menyusun `best_model.joblib` & `model_metadata.json`

In [ ]:
best_pipeline = Pipeline([("scaler", scaler), ("kmeans", best_model)])

bundle = {
    "pipeline": best_pipeline,          # pipeline.predict(df[features]) -> cluster
    "pca": pca,                         # untuk koordinat scatter di web
    "features": FEATURES,
    "algorithm": "KMeans",
    "model_version": "1.0.0",
    "sklearn_version": sklearn.__version__,
}

X_pca_centers = pca.transform(best_model.cluster_centers_)
sample_idx = np.random.default_rng(RANDOM_STATE).choice(len(X), size=min(600, len(X)), replace=False)

clusters = []
for c in range(N_CLUSTERS):
    m = labels == c
    e = {"cluster_id": c, "name": cluster_names[c], "size": int(m.sum()), "share": float(m.mean()),
         "centroid": {f: float(centroids_orig.loc[c, f]) for f in FEATURES},
         "centroid_z": {f: float(centroids_z.loc[c, f]) for f in FEATURES},
         "distance_threshold": thr_km[c],
         "pca_centroid": [float(v) for v in X_pca_centers[c]]}
    if "process_step" in df.columns:
        e["process_step_distribution"] = df.loc[m, "process_step"].value_counts(normalize=True).round(4).to_dict()
    if y_defect is not None:
        e["defect_count"] = int(y_defect[m].sum()); e["defect_rate"] = float(y_defect[m].mean())
    clusters.append(e)

def clean(v):
    return None if isinstance(v, float) and np.isnan(v) else v

metadata = {
    "model_name": "SmartWafer Clustering",
    "model_version": bundle["model_version"],
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "model_file": "best_model.joblib",
    "best_algorithm": best_row.algoritma,
    "best_config": best_row.konfigurasi,
    "selection_rule": "kandidat & dapat memprediksi data baru & >=2 cluster, lalu silhouette tertinggi",
    "features": FEATURES,
    "feature_labels": FEATURE_LABELS,
    "excluded_columns": {"defect_label": "label (hanya evaluasi eksternal)", "wafer_id": "identifier",
                         "electrical_power_mw": "redundan (V x I)", "process_step": "nominal; LabelEncoder = artefak"},
    "n_clusters": N_CLUSTERS,
    "random_state": RANDOM_STATE,
    "metrics": {k: clean(float(best_row[k])) for k in ["silhouette", "davies_bouldin", "calinski_harabasz"]},
    "stability": {"method": "ARI vs model utama, random seed berbeda", "seeds": STABILITY_SEEDS,
                  "ari": [round(float(a), 4) for a in ari_seeds], "ari_mean": round(float(np.mean(ari_seeds)), 4)},
    "k_evaluation": k_eval.round(6).replace({np.nan: None}).to_dict(orient="records"),
    "anomaly": {"method": "jarak euclidean ke centroid (ruang ter-scaling)", "percentile": ANOMALY_PERCENTILE},
    "external_evaluation": ext["K-Means"] or None,
    "model_comparison": [{k: clean(v) if not isinstance(v, (np.bool_, bool)) else bool(v) for k, v in r.items()}
                         for r in comparison.to_dict(orient="records")],
    "feature_stats": {f: {"mean": float(X_raw[f].mean()), "std": float(X_raw[f].std()),
                          "min": float(X_raw[f].min()), "max": float(X_raw[f].max())} for f in FEATURES},
    "pca": {"explained_variance_ratio": [float(v) for v in pca.explained_variance_ratio_]},
    "pca_sample": [{"x": float(X_pca[i, 0]), "y": float(X_pca[i, 1]), "cluster": int(labels[i])} for i in sample_idx],
    "clusters": clusters,
}

print(json.dumps({k: v for k, v in metadata.items() if k not in ("pca_sample", "k_evaluation", "model_comparison")},
                 indent=2, ensure_ascii=False, default=str)[:2500])

## 17. Export 3 file output

In [ ]:
joblib.dump(bundle, os.path.join(OUTPUT_DIR, "best_model.joblib"))
with open(os.path.join(OUTPUT_DIR, "model_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
# model_comparison.csv sudah disimpan di bagian 13

for name in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, name)
    print(f"{name:25s} {os.path.getsize(p)/1024:8.1f} KB")
assert sorted(os.listdir(OUTPUT_DIR)) == ["best_model.joblib", "model_comparison.csv", "model_metadata.json"]

## 18. Uji reload & inference (simulasi aplikasi)

Memuat ulang ketiga file dari disk, persis seperti yang dilakukan aplikasi.

In [ ]:
def predict_cluster(sample: dict, out_dir: str = OUTPUT_DIR) -> dict:
    b = joblib.load(os.path.join(out_dir, "best_model.joblib"))
    with open(os.path.join(out_dir, "model_metadata.json"), encoding="utf-8") as f:
        meta = json.load(f)
    x = pd.DataFrame([[float(sample[f]) for f in b["features"]]], columns=b["features"])
    sc, km = b["pipeline"].named_steps["scaler"], b["pipeline"].named_steps["kmeans"]
    xs = sc.transform(x)
    cid = int(km.predict(xs)[0])
    dist = float(np.linalg.norm(xs[0] - km.cluster_centers_[cid]))
    info = meta["clusters"][cid]
    xy = b["pca"].transform(xs)[0]
    return {"cluster_id": cid, "cluster_name": info["name"],
            "distance_to_centroid": round(dist, 4), "distance_threshold": round(info["distance_threshold"], 4),
            "is_anomaly": dist > info["distance_threshold"],
            "pca": {"x": round(float(xy[0]), 4), "y": round(float(xy[1]), 4)}}

# Uji 1: konsistensi seluruh dataset
b = joblib.load(os.path.join(OUTPUT_DIR, "best_model.joblib"))
assert (b["pipeline"].predict(df[b["features"]]) == labels).all()
print("Seluruh", len(df), "baris konsisten setelah reload ✔")

# Uji 2: model_comparison.csv terbaca & tepat satu model terpilih
cmp_ = pd.read_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"))
assert cmp_["terpilih"].sum() == 1
print("model_comparison.csv:", len(cmp_), "baris, terpilih =", cmp_.loc[cmp_.terpilih, "algoritma"].item(), "✔")

# Uji 3: input normal & kondisi ekstrem (tekanan rendah, suhu tinggi — mirip pola defect)
print("Normal :", predict_cluster(dict(zip(FEATURES, [450, 760, 120, 95, 5.0, 20.0]))))
print("Ekstrem:", predict_cluster(dict(zip(FEATURES, [490, 670, 120, 95, 5.0, 20.0]))))

## 19. Download output

In [ ]:
zip_path = shutil.make_archive("clustering_output", "zip", OUTPUT_DIR)
print("Zip dibuat:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Bukan di Colab: ambil file zip langsung dari folder kerja.")

## 20. Ringkasan untuk laporan

| Item | Nilai (lihat output sel) |
|---|---|
| Algoritma dibandingkan | K-Means, Agglomerative (Ward), DBSCAN (+ eksperimen 7 fitur) |
| Model terpilih | K-Means K=3 — aturan: bisa prediksi data baru, ≥2 cluster, silhouette tertinggi |
| Metrik | Silhouette / DBI / CH di `model_comparison.csv` |
| Stabilitas | rata-rata ARI antar seed (bagian 11) |
| Anomali | P95 jarak ke centroid; x/7 defect tertangkap |

**Keterbatasan:** silhouette rendah (~0.1) dan stabilitas antar seed rendah menunjukkan data tidak memiliki cluster alami; cluster adalah **segmentasi rezim operasi**. DBSCAN hanya menemukan satu cluster padat + noise, yang menguatkan kesimpulan tersebut. Nilai praktis utama ada pada **deteksi kondisi tidak lazim**, yang berkorelasi dengan wafer defect (sampel defect sangat kecil).